# 2D Natural Cubic Spline Interpolation Demo

This notebook demonstrates 2-dimensional natural cubic spline interpolation using the `ndim_spline_jax` library.
We build a spline from sampled data on a 2D grid, evaluate it, compute gradients via JAX autodiff, and benchmark JIT-compiled calls.

In [ ]:
import numpy as np
import jax.numpy as jnp
from jax import jit, grad, value_and_grad
from ndim_spline_jax import compute_coefs, make_interpolant

## 1. Define grid and generate data

In [ ]:
# Domain bounds, number of intervals per axis
a = [0.0, 0.0]
b = [1.0, 2.0]
n = [10, 10]

# Grid points
x1 = np.linspace(a[0], b[0], n[0] + 1)
x2 = np.linspace(a[1], b[1], n[1] + 1)

# Meshgrid (indexing='ij' for tensor layout)
X1, X2 = np.meshgrid(x1, x2, indexing='ij')

# Data: sin(x1) * sin(x2)
y_data = np.sin(X1) * np.sin(X2)

print("y_data shape:", y_data.shape)

## 2. Compute spline coefficients

In [ ]:
c = compute_coefs(2, jnp.array(y_data))
print("Coefficient tensor shape:", c.shape)

## 3. Create interpolant

In [ ]:
s = make_interpolant(a, b, n, c)

## 4. Evaluate, differentiate, and JIT-compile

In [ ]:
# Test point (midpoint of domain)
x = jnp.array([0.5, 1.0])

# Evaluate
val = s(x)
print("s(x)          =", val)

# Gradient
grad_val = grad(s)(x)
print("grad(s)(x)    =", grad_val)

# Value and gradient together
v, g = value_and_grad(s)(x)
print("value_and_grad:", v, g)

# JIT-compiled evaluation
val_jit = jit(s)(x)
print("jit(s)(x)     =", val_jit)

# JIT-compiled gradient
grad_jit = jit(grad(s))(x)
print("jit(grad(s))  =", grad_jit)

# JIT-compiled value and gradient
v_jit, g_jit = jit(value_and_grad(s))(x)
print("jit(vg)       =", v_jit, g_jit)

## 5. Benchmark

In [ ]:
import timeit

s_jit = jit(s)
ds_jit = jit(grad(s))
vg_jit = jit(value_and_grad(s))

# Warm up
_ = s_jit(x)
_ = ds_jit(x)
_ = vg_jit(x)

%timeit s_jit(x)
%timeit ds_jit(x)
%timeit vg_jit(x)